# HuggingFace Embeddings

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- HuggingFace(허깅페이스)의 Endpoint 방식과 Local 방식의 차이를 설명할 수 있어요
- `multilingual-e5-large-instruct`와 `BGE-M3` 모델로 다국어 임베딩을 생성할 수 있어요
- 상황에 맞는 임베딩 모델을 선택할 수 있어요

## 사전 지식

- 노트북 01(OpenAI Embeddings) 학습 완료
- HuggingFace API 토큰 (Endpoint 방식 사용 시)

---

## HuggingFace Embeddings란?

HuggingFace는 수천 개의 오픈소스 임베딩 모델을 제공하는 플랫폼이에요. API 비용 없이 무료로 사용할 수 있고, 데이터가 외부로 전송되지 않아 프라이버시를 보장할 수 있어요.

```mermaid
flowchart LR
    T["텍스트 입력"]:::input
    subgraph HF["HuggingFace 모델"]
        E["Endpoint<br/>(클라우드 API)"]:::process
        L["Local<br/>(로컬 실행)"]:::process
    end
    V["임베딩 벡터<br/>1024차원"]:::output

    T --> E --> V
    T --> L --> V

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 주요 모델 비교

| 모델 | 차원 | 언어 | 특징 |
|------|------|------|------|
| `multilingual-e5-large-instruct` | 1024 | 다국어 | 지시문 이해 능력 |
| `BAAI/bge-m3` | 1024 | 다국어 | Dense + Sparse + Multi-Vector |
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | 영어 | 빠르고 가벼움 |

> **실무 팁**: 한국어 문서가 포함된 RAG 시스템에는 `multilingual-e5-large-instruct` 또는 `BAAI/bge-m3`를 사용하는 게 좋아요.

## 환경 설정

모델 파일은 `./cache/` 경로에 다운로드돼요. 처음 실행 시 시간이 걸릴 수 있어요.

In [1]:
from dotenv import load_dotenv
import os
import warnings

# 경고 무시
warnings.filterwarnings("ignore")

# API 키 정보 로드
load_dotenv()

# HuggingFace 모델을 ./cache/ 경로에 다운로드
os.environ["HF_HOME"] = "./cache/"


## 샘플 데이터 준비

한국어와 영어가 섞인 다국어 텍스트로 임베딩 성능을 테스트해볼게요.

In [2]:
texts = [
    "딥러닝 모델의 성능을 향상시키기 위한 다양한 기법들이 연구되고 있습니다.",
    "Deep learning techniques have revolutionized computer vision and natural language processing.",
    "신경망 구조를 최적화하여 학습 속도와 정확도를 높일 수 있습니다.",
    "Transfer learning allows models to leverage knowledge from pre-trained networks.",
    "자연어 처리 분야에서 트랜스포머 아키텍처가 큰 발전을 이끌었습니다.",
]


---

## 1. HuggingFace Endpoint Embeddings

HuggingFace 클라우드 API로 임베딩을 생성하는 방식이에요. 로컬 GPU 없이도 사용할 수 있고 모델 다운로드도 필요 없어요.

**장점**: 빠른 시작, 로컬 리소스 불필요
**단점**: API 호출 비용, 인터넷 연결 필요

> 🎯 **강의 포인트**: HuggingFace는 **Endpoint 방식**(클라우드 API)과 **Local 방식**(직접 실행) 두 가지예요. 처음 시작하는 팀은 Endpoint로 빠르게 프로토타입을 만들고, 이후 비용·프라이버시 이슈가 생기면 Local로 전환하는 전략을 많이 사용해요.

> ⚠️ **자주 하는 실수**: `HUGGINGFACEHUB_API_TOKEN`이 `.env` 파일에 설정되어 있어야 해요. [HuggingFace 토큰 발급](https://huggingface.co/settings/tokens)에서 무료로 발급받을 수 있어요.

In [3]:

# ---------------------------------------------------
# HuggingFace Endpoint 방식으로 임베딩 모델 초기화
# ---------------------------------------------------

# ============================================================
# TODO: HuggingFaceEndpointEmbeddings로 클라우드 임베딩 모델을 초기화해보세요
# 힌트: model, task, huggingfacehub_api_token 파라미터를 설정해요
# 예상 결과: "모델 초기화 완료" 출력
# ============================================================

from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

# HuggingFaceEndpointEmbeddings: HuggingFace 클라우드 API를 통한 임베딩
# task="feature-extraction": 텍스트를 벡터로 추출하는 태스크
hf_endpoint_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task="feature-extraction",
    # huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
)

print(f"✅ 모델 초기화 완료: {model_name}")
print("  - 방식: HuggingFace API (클라우드)")


✅ 모델 초기화 완료: intfloat/multilingual-e5-large-instruct
  - 방식: HuggingFace API (클라우드)


In [4]:

# ---------------------------------------------------
# 문서 배치 임베딩 및 벡터 차원 확인
# ---------------------------------------------------

# embed_documents(): 여러 문서를 한 번에 임베딩 (배치 처리)
embedded_documents = hf_endpoint_embeddings.embed_documents(texts)

print(f"[HuggingFace Endpoint Embedding]")
print(f"모델: \t\t{model_name}")
print(f"문서 수: \t{len(embedded_documents)}")
print(f"벡터 차원: \t{len(embedded_documents[0])}")
print(f"벡터 일부: \t{embedded_documents[0][:5]}")


[HuggingFace Endpoint Embedding]
모델: 		intfloat/multilingual-e5-large-instruct
문서 수: 	5
벡터 차원: 	1024
벡터 일부: 	[0.00034766149474307895, 0.024516623467206955, 0.0011223346227779984, -0.023458929732441902, 0.03127119690179825]


### 1.1 유사도 계산

쿼리와 문서 간의 유사도를 계산해서 검색을 수행해볼게요.

In [5]:

# ---------------------------------------------------
# 벡터 내적으로 쿼리-문서 유사도 계산 및 순위 출력
# ---------------------------------------------------

# ============================================================
# TODO: 쿼리를 임베딩하고 문서와의 유사도를 계산해보세요
# 힌트: np.array(query) @ np.array(docs).T 로 내적 유사도 계산
# 예상 결과: 유사도 순으로 정렬된 문서 목록 출력
# ============================================================

import numpy as np

# 1단계: 검색 쿼리 임베딩
query = "딥러닝 모델 최적화 방법"
embedded_query = hf_endpoint_embeddings.embed_query(query)

# 2단계: 벡터 내적을 통한 유사도 계산
# 내적(dot product): normalize된 벡터 간 내적 = 코사인 유사도와 동일
similarities = np.array(embedded_query) @ np.array(embedded_documents).T

# 3단계: 유사도 내림차순 정렬
sorted_indices = similarities.argsort()[::-1]

print(f"[쿼리] {query}")
print("=" * 60)
for i, idx in enumerate(sorted_indices, 1):
    print(f"[{i}위] 유사도: {similarities[idx]:.4f}")
    print(f"    {texts[idx]}")
    print()


[쿼리] 딥러닝 모델 최적화 방법
[1위] 유사도: 0.8918
    딥러닝 모델의 성능을 향상시키기 위한 다양한 기법들이 연구되고 있습니다.

[2위] 유사도: 0.8795
    신경망 구조를 최적화하여 학습 속도와 정확도를 높일 수 있습니다.

[3위] 유사도: 0.7953
    자연어 처리 분야에서 트랜스포머 아키텍처가 큰 발전을 이끌었습니다.

[4위] 유사도: 0.7888
    Deep learning techniques have revolutionized computer vision and natural language processing.

[5위] 유사도: 0.7807
    Transfer learning allows models to leverage knowledge from pre-trained networks.



---

## 2. HuggingFace Local Embeddings

로컬 환경에서 모델을 직접 실행하는 방식이에요. 처음 실행 시 모델 파일이 다운로드되고, 이후에는 오프라인에서도 사용할 수 있어요.

**장점**: API 비용 없음, 프라이버시 보장, 오프라인 사용 가능
**단점**: 모델 다운로드 시간, GPU 권장 (CPU도 가능하지만 느림)

> 💡 **실무 팁**: `device` 설정을 환경에 맞게 바꿔주세요. Mac M 시리즈는 `"mps"`, NVIDIA GPU는 `"cuda"`, 그 외에는 `"cpu"`를 사용해요.

> 🔑 **핵심 개념**: `normalize_embeddings=True`는 **반드시** 설정해야 하는 옵션이에요. 벡터를 정규화하면 내적(dot product)으로 코사인 유사도와 동일한 결과를 얻을 수 있고, VectorStore의 검색 정확도가 향상돼요.

In [6]:

# ---------------------------------------------------
# HuggingFace Local 방식으로 임베딩 모델 초기화
# ---------------------------------------------------

# ============================================================
# TODO: HuggingFaceEmbeddings로 로컬 실행 임베딩 모델을 초기화해보세요
# 힌트: model_kwargs={"device": "cpu"}, encode_kwargs={"normalize_embeddings": True}
# 예상 결과: "모델 초기화 완료" 출력 (처음 실행 시 모델 다운로드 발생)
# ============================================================

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# 로컬에서 실행할 모델 설정
model_name = "intfloat/multilingual-e5-large-instruct"

# HuggingFaceEmbeddings: 모델 파일을 로컬에 다운로드해 직접 실행
# model_kwargs={"device": "cpu"}: 실행 장치 설정 (cuda/mps/cpu)
# encode_kwargs={"normalize_embeddings": True}: 벡터 정규화 — 코사인 유사도 계산에 최적
hf_local_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},  # GPU 사용 시: "cuda" 또는 "mps"
    encode_kwargs={"normalize_embeddings": True},  # 정규화로 코사인 유사도 최적화
)

print(f"✅ 모델 초기화 완료: {model_name}")
print("  - 방식: 로컬 실행")
print("  - 정규화: True (코사인 유사도에 최적)")


✅ 모델 초기화 완료: intfloat/multilingual-e5-large-instruct
  - 방식: 로컬 실행
  - 정규화: True (코사인 유사도에 최적)


In [7]:
# 문서 임베딩
local_embedded_documents = hf_local_embeddings.embed_documents(texts)

print(f"\n[HuggingFace Local Embedding]")
print(f"모델: \t\t{model_name}")
print(f"벡터 차원: \t{len(local_embedded_documents[0])}")
print(f"벡터 일부: \t{local_embedded_documents[0][:5]}")



[HuggingFace Local Embedding]
모델: 		intfloat/multilingual-e5-large-instruct
벡터 차원: 	1024
벡터 일부: 	[0.0003476527926977724, 0.02451658621430397, 0.0011222852626815438, -0.02345903031527996, 0.03127126395702362]


---

## 3. BGE-M3: 강력한 다국어 임베딩

BGE-M3(BAAI General Embedding - Multi-lingual, Multi-function, Multi-granularity)는 BAAI(베이징 인공지능 연구원)에서 개발한 다국어 임베딩 모델이에요.

**세 가지 임베딩 방식**을 하나의 모델로 지원해요:
- **Dense Vector**(밀집 벡터): 일반적인 의미 기반 임베딩
- **Sparse Vector**(희소 벡터): 키워드 정확 매칭 (TF-IDF와 유사)
- **Multi-Vector**(멀티 벡터): ColBERT 방식의 토큰 단위 세밀한 매칭

> 🎯 **강의 포인트**: BGE-M3의 가장 큰 강점은 **하이브리드 검색**이에요. Dense만 쓰면 키워드가 정확히 일치해도 놓치는 경우가 있고, Sparse만 쓰면 의미적으로 유사한 문서를 못 찾아요. 두 방식을 조합하면 검색 정확도가 크게 높아져요.

In [8]:

# ---------------------------------------------------
# BGE-M3 모델로 임베딩 초기화
# ---------------------------------------------------

# BGE-M3(BAAI/bge-m3): Dense·Sparse·Multi-Vector 세 가지 방식을 하나로 통합한 다국어 모델
bge_model_name = "BAAI/bge-m3"

bge_embeddings = HuggingFaceEmbeddings(
    model_name=bge_model_name,
    model_kwargs={"device": "cpu"},  # GPU 사용 시: "cuda" 또는 "mps"
    encode_kwargs={"normalize_embeddings": True},
)

print(f"✅ BGE-M3 모델 초기화 완료")
print(f"  - 모델: {bge_model_name}")
print(f"  - 특징: Dense, Sparse, Multi-Vector 지원")


✅ BGE-M3 모델 초기화 완료
  - 모델: BAAI/bge-m3
  - 특징: Dense, Sparse, Multi-Vector 지원


In [9]:

# ---------------------------------------------------
# BGE-M3로 문서 임베딩 및 검색 결과 확인
# ---------------------------------------------------

# BGE-M3로 문서 임베딩 (사용법은 다른 HuggingFace 모델과 동일)
bge_embedded_documents = bge_embeddings.embed_documents(texts)

print(f"\n[BGE-M3 Embedding]")
print(f"모델: \t\t{bge_model_name}")
print(f"벡터 차원: \t{len(bge_embedded_documents[0])}")

# 검색 테스트: 쿼리 임베딩 후 내적으로 유사도 계산
bge_embedded_query = bge_embeddings.embed_query(query)
bge_similarities = np.array(bge_embedded_query) @ np.array(bge_embedded_documents).T
bge_sorted_indices = bge_similarities.argsort()[::-1]

print(f"\n[쿼리] {query}")
print("=" * 60)
for i, idx in enumerate(bge_sorted_indices[:3], 1):
    print(f"[{i}위] 유사도: {bge_similarities[idx]:.4f}")
    print(f"    {texts[idx]}")
    print()



[BGE-M3 Embedding]
모델: 		BAAI/bge-m3
벡터 차원: 	1024

[쿼리] 딥러닝 모델 최적화 방법
[1위] 유사도: 0.7280
    딥러닝 모델의 성능을 향상시키기 위한 다양한 기법들이 연구되고 있습니다.

[2위] 유사도: 0.6172
    신경망 구조를 최적화하여 학습 속도와 정확도를 높일 수 있습니다.

[3위] 유사도: 0.4714
    Transfer learning allows models to leverage knowledge from pre-trained networks.



### 3.1 FlagEmbedding으로 고급 기능 활용 (선택)

`FlagEmbedding` 라이브러리를 사용하면 BGE-M3의 세 가지 임베딩 방식을 모두 활용할 수 있어요. LangChain 통합보다 세밀한 제어가 필요할 때 사용해요.

> **주의**: 아래 코드는 `FlagEmbedding` 패키지가 필요해요 (`pip install FlagEmbedding`). 일반 RAG 사용에는 앞에서 배운 `HuggingFaceEmbeddings`만으로 충분해요.

---

## 4. 모델 선택 가이드

상황에 따라 적합한 모델이 달라요. 아래 기준으로 선택해보세요.

| 상황 | 추천 모델 | 이유 |
|------|-----------|------|
| 빠른 프로토타입, 영어 전용 | `all-MiniLM-L6-v2` | 384차원, 가장 빠름 |
| 한국어 포함 다국어 | `multilingual-e5-large-instruct` | 1024차원, 다국어 최적화 |
| 최고 성능 + 하이브리드 검색 | `BAAI/bge-m3` | Dense+Sparse+Multi-Vector |
| 비용 최소화 (유료) | `text-embedding-3-small` | OpenAI, 빠르고 저렴 |

> 💡 **실무 팁**: 모델 선택은 **MTEB 리더보드**(https://huggingface.co/spaces/mteb/leaderboard)를 참고하세요. 특히 한국어 태스크(Ko-MTEB) 점수를 확인하면 한국어 RAG에 맞는 모델을 고를 수 있어요.

---

## 마무리

### 핵심 요약

- **Endpoint 방식**: HuggingFace 클라우드 API — API 토큰 필요, 빠른 시작
- **Local 방식**: 로컬 모델 실행 — 무료, 프라이버시, `device` 설정 중요
- **BGE-M3**: Dense + Sparse + Multi-Vector 세 가지 방식 모두 지원
- **`normalize_embeddings=True`**: 코사인 유사도 사용 시 필수 설정

### 다음 학습

**6-3 노트북 03**: 한국어 특화 임베딩 — Upstage Solar 모델로 한국어 검색 시스템의 Query/Passage 비대칭 구조를 배워볼게요.